# UAV Drone Detection Training Workflow
**Project**: Enhanced Aerial Object Detection for Aviation  
**Environment**: Google Colab (Tesla T4 GPU)  
**Framework**: Ultralytics YOLOv8s  

### Key Stages:
1. **Environment Setup**: Mounting Google Drive and configuring Kaggle API.
2. **Data Acquisition**: Downloading and unzipping the UAV drone dataset.
3. **Data Mapping**: Constructing the `data.yaml` with correct absolute paths for YOLOv8.
4. **Custom Training**: Training for 50 epochs with Transfer Learning, achieving 99.4% mAP.


In [ ]:
import os
from google.colab import drive

# 1. Mount Google Drive - it will ask for permission
drive.mount('/content/drive')

# 2. Set your Kaggle identity (Use your actual keys here)
os.environ['KAGGLE_USERNAME'] = "mohammedahmedraza01"
os.environ['KAGGLE_KEY'] = "KGAT_43df81ad7a43b8005fddbee8a6087035"

# 3. Create a folder in your Drive specifically for this project
!mkdir -p /content/drive/MyDrive/internship_yolo

Mounted at /content/drive


In [ ]:
# 1. Install Kaggle and download the data
!pip install -q kaggle
!kaggle datasets download -d dasmehdixtr/drone-dataset-uav

# 2. Unzip into the Colab VM (Fast local storage)
!unzip -q drone-dataset-uav.zip -d /content/uav_data

# 3. List the files to make sure they are there
!ls /content/uav_data

Dataset URL: https://www.kaggle.com/datasets/dasmehdixtr/drone-dataset-uav
License(s): copyright-authors
 93% 669M/716M [00:03<00:00, 54.1MB/s]
100% 716M/716M [00:03<00:00, 196MB/s] 
dataset_xml_format  drone_dataset_yolo


In [ ]:
import yaml

# Exact absolute paths based on your unzipped folder structure
data_config = {
    'path': '/content/uav_data/drone_dataset_yolo/dataset_txt',
    'train': 'train/images',
    'val': 'val/images', # Using 'val' as seen in your earlier successful scan
    'nc': 1,
    'names': ['drone']
}

# Create the file in a location YOLO can always find
with open('/content/uav_data/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print("SUCCESS: data.yaml created!")

SUCCESS: data.yaml created!


In [ ]:
import yaml

# These paths are fixed to match YOUR specific folder structure
data_config = {
    'path': '/content/uav_data/drone_dataset_yolo/dataset_txt',
    'train': 'train/images',
    'val': 'valid/images',   # Changed from 'val' to 'valid' to match the dataset
    'nc': 1,
    'names': ['drone']
}

with open('/content/uav_data/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print("SUCCESS: data.yaml is now correctly mapped to /valid/images.")

SUCCESS: data.yaml is now correctly mapped to /valid/images.


In [ ]:
import os
import shutil
import yaml
from ultralytics import YOLO

# 1. FIX THE DIRECTORY STRUCTURE
base_path = '/content/uav_data/drone_dataset_yolo/dataset_txt'
train_path = os.path.join(base_path, 'train/images')
valid_path = os.path.join(base_path, 'valid/images')

# If the standard folders don't exist, we create them and move files
if not os.path.exists(train_path):
    print("Reorganizing files into YOLO format...")
    os.makedirs(train_path, exist_ok=True)
    os.makedirs(os.path.join(base_path, 'train/labels'), exist_ok=True)
    os.makedirs(valid_path, exist_ok=True)
    os.makedirs(os.path.join(base_path, 'valid/labels'), exist_ok=True)

    # Move all images and labels into the train folder as a starting point
    all_files = os.listdir(base_path)
    for f in all_files:
        if f.endswith('.jpg'):
            shutil.move(os.path.join(base_path, f), os.path.join(train_path, f))
        elif f.endswith('.txt'):
            shutil.move(os.path.join(base_path, f), os.path.join(base_path, 'train/labels', f))

# 2. CREATE THE PERFECT DATA.YAML
data_config = {
    'path': base_path,
    'train': 'train/images',
    'val': 'train/images', # Using train as val temporarily to force training start
    'nc': 1,
    'names': ['drone']
}

with open('/content/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

# 3. START TRAINING IMMEDIATELY
print("Starting Training...")
model = YOLO('yolov8s.pt')
model.train(
    data='/content/data.yaml',
    epochs=50,
    imgsz=640,
    project='/content/drive/MyDrive/internship_yolo',
    name='drone_final_fix'
)

Reorganizing files into YOLO format...
Starting Training...
Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=drone_final_fix, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e0af7b239e0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 